# 03_pc_<agreement>_<source>_to_<target>
Run-all-safe enforcement pipeline notebook.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit import (
    load_fabric_config, validate_notebook_name, assert_notebook_name_valid,
    bootstrap_fabric_env, generate_run_id, build_runtime_context, load_data_contract,
    lakehouse_table_read, warehouse_read, get_path, run_quality_rules,
    add_datetime_features, add_audit_columns, add_hash_columns, default_technical_columns,
    lakehouse_table_write, warehouse_write, generate_metadata_profile,
    profile_dataframe_to_metadata, build_quality_result_records, build_dataset_run_record,
    write_metadata_records, build_lineage_from_notebook_code, build_lineage_handover_markdown,
    build_lineage_records
)


In [ ]:
ENV_NAME = "dev"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"
TARGET_KIND = "lakehouse"
SOURCE_TABLE = "TODO_source_table"
TARGET_TABLE = "TODO_target_table"
CONTRACT_PATH = "contracts/examples/normalized_data_product_contract.yml"
WRITE_MODE = "overwrite"


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])
RUN_ID = generate_run_id(prefix="pc")
runtime_context = build_runtime_context(dataset_name="target_dataset", environment=ENV_NAME, source_table=SOURCE_TABLE, target_table=TARGET_TABLE, run_id=RUN_ID)


In [ ]:
contract = load_odcs_contract(CONTRACT_PATH)
contract_errors = validate_odcs_contract(contract)
if contract_errors:
    raise ValueError("ODCS contract validation failed: " + " | ".join(contract_errors))
source_contract_summary = build_source_input_contract_summary(contract, object_name=SOURCE_TABLE)
print("Loaded ODCS source summary:", source_contract_summary)


In [ ]:
if SOURCE_KIND == "lakehouse":
    source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
required_columns = extract_required_columns(contract, object_name=SOURCE_TABLE)
business_keys = extract_business_keys(contract, object_name=SOURCE_TABLE)
quality_rules = map_odcs_quality_rules_to_fabricops_rules(contract, object_name=SOURCE_TABLE)
missing = sorted(set(required_columns) - set(df_source.columns))
if missing:
    raise ValueError(f"Fail fast: missing required source columns: {missing}")


In [ ]:
# TODO: approved deterministic transformation logic only
df_transformed = df_source


In [ ]:
runtime_quality_rules = [rule for rule in quality_rules if not rule.get("skipped")]
dq_results = run_quality_rules(
    df_transformed,
    runtime_quality_rules,
    dataset_name=DATASET_NAME,
    table_name=SOURCE_TABLE,
)
if not dq_results.get("can_continue", True):
    raise ValueError("Fail fast: ODCS quality gate failed.")


In [ ]:
DATETIME_COLUMN = None
DATETIME_PREFIX = None
if DATETIME_COLUMN:
    df_transformed = add_datetime_features(
        df_transformed,
        datetime_column=DATETIME_COLUMN,
        prefix=DATETIME_PREFIX,
    )


In [ ]:
BUSINESS_KEYS = business_keys
df_standard = add_audit_columns(df_transformed, run_id=RUN_ID, environment=ENV_NAME, source_table=SOURCE_TABLE)
if BUSINESS_KEYS:
    df_standard = add_hash_columns(
        df_standard,
        business_keys=BUSINESS_KEYS,
    )


In [ ]:
required_output_columns = set(contract.target.required_columns or [])
actual_output_cols = set(c for c in df_standard.columns if c not in tech_cols)
if required_output_columns and required_output_columns != actual_output_cols:
    raise ValueError("Fail fast: output contract mismatch.")


In [ ]:
if TARGET_KIND == "lakehouse":
    target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
    lakehouse_table_write(df_standard, target_path, TARGET_TABLE, mode=WRITE_MODE)
else:
    warehouse_write(df_standard, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)


In [ ]:
if TARGET_KIND == "lakehouse":
    target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
    df_written = lakehouse_table_read(target_path, TARGET_TABLE)
else:
    df_written = warehouse_read(env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, config=CONFIG)
if df_written.count() == 0:
    raise ValueError("Fail fast: target write produced zero rows.")


In [ ]:
output_profile = generate_metadata_profile(df_written, dataset_name="target_dataset")
output_profile_rows = profile_dataframe_to_metadata(df_written, dataset_name="target_dataset")


In [ ]:
quality_records = build_quality_result_records(dq_results, dataset_name="target_dataset", run_id=RUN_ID)
dataset_record = build_dataset_run_record(dataset_name="target_dataset", run_id=RUN_ID, row_count=df_written.count())
write_metadata_records([*quality_records, dataset_record, *output_profile_rows], config=CONFIG)


In [ ]:
lineage = build_lineage_from_notebook_code(__file__, dataset_name="target_dataset", run_id=RUN_ID)
lineage_records = build_lineage_records(dataset_name="target_dataset", run_id=RUN_ID, lineage_steps=lineage.get("steps", []))
print(build_lineage_handover_markdown(dataset_name="target_dataset", run_id=RUN_ID, lineage_result=lineage))
